# Shopping Mall Data Cleaning

This notebook queries OpenStreetMap through the Overpass API and produces a cleaned list of shopping malls in Singapore.

## Cleaning objective
- query mall features within Singapore's administrative boundary
- extract names and representative coordinates
- remove duplicate mall names
- export the cleaned dataset to the project `data/cleaned` folder


In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path

import pandas as pd
import requests

OUTPUT_FILE = Path("../../data/clean/mall_cleaned.csv")


## Query OpenStreetMap mall records

The helper function below retrieves nodes, ways, and relations tagged as malls, then standardizes them to one row per mall with latitude and longitude coordinates.


In [5]:
# ============================================================
# 2. Helper function
# ============================================================

def get_shopping_malls() -> pd.DataFrame:
    """Fetch shopping mall records in Singapore from the Overpass API."""
    overpass_url = "https://overpass-api.de/api/interpreter"

    # Search both `shop=mall` and `building=mall` tags within Singapore.
    query = """
    [out:json][timeout:120];
    area["ISO3166-1"="SG"][admin_level=2]->.sg;
    (
      node["shop"="mall"](area.sg);
      way["shop"="mall"](area.sg);
      relation["shop"="mall"](area.sg);
      node["building"="mall"](area.sg);
      way["building"="mall"](area.sg);
      relation["building"="mall"](area.sg);
    );
    out center tags;
    """

    response = requests.post(overpass_url, data={"data": query}, timeout=180)
    response.raise_for_status()
    data = response.json()

    rows = []
    for element in data.get("elements", []):
        tags = element.get("tags", {})
        name = tags.get("name")
        if not name:
            continue

        # Nodes expose `lat` and `lon` directly, while ways and relations
        # return a representative centroid under `center`.
        lat = element.get("lat", element.get("center", {}).get("lat"))
        lon = element.get("lon", element.get("center", {}).get("lon"))

        if lat is not None and lon is not None:
            rows.append({"mall_name": name, "lat": lat, "lon": lon})

    return (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["mall_name"])
        .sort_values("mall_name")
        .reset_index(drop=True)
    )


## Create cleaned output

This step runs the query, previews the cleaned dataframe, and writes the final dataset to disk.


In [6]:
# ============================================================
# 3. Build cleaned dataset
# ============================================================

mall_df = get_shopping_malls()
print("Shopping mall records:", len(mall_df))
mall_df.head()


Shopping mall records: 228


,mall_name,lat,lon
0,100AM,1.274904,103.843614
1,18 Tai Seng,1.336297,103.888832
2,313@Somerset,1.301018,103.838287
3,888 Plaza,1.437814,103.795290
4,Admiralty Place,1.440258,103.801298


In [7]:
# ============================================================
# 4. Save cleaned dataset
# ============================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
mall_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned mall data to {OUTPUT_FILE.resolve()}")


Saved cleaned mall data to /Users/celine/Documents/GitHub/DSA4264-Public-Policy-and-Society/data/cleaned/mall_cleaned.csv
